# 🏈 NFL Pick Em — BigQuery ETL
Run each cell **in order**, top to bottom.

Confirmed against `2025_NFL_Pick_Em.xlsx`:
- Sheet: `Picks`
- Header row: row 4 in Excel = index 3
- 30 players, Week 1 through Week 18
- Uses batch load jobs (free Sandbox tier compatible)

In [ ]:
# ── CELL 1: Authenticate ──────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print('Authenticated!')

In [ ]:
# ── CELL 2: Install dependencies ──────────────────────────────────
!pip install google-cloud-bigquery openpyxl pyarrow --quiet
print('Dependencies ready!')

In [ ]:
# ── CELL 3: Upload your Excel file ────────────────────────────────
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    print(f'Uploaded: {filename}')

In [ ]:
# ── CELL 4: Config ────────────────────────────────────────────────
PROJECT_ID  = 'nfl-pick-em-493506'
DATASET_ID  = 'nfl_pickem'
SEASON_YEAR = 2025
EXCEL_FILE  = '2025_NFL_Pick_Em.xlsx'  # exact filename from Cell 3
SHEET_NAME  = 'Picks'
HEADER_ROW  = 3   # 0-indexed; row 4 in Excel = index 3
print('Config set!')

In [ ]:
# ── CELL 5: Preview the data (no BigQuery yet) ────────────────────
# Run this to confirm the file looks right before loading anything.
import pandas as pd

df_raw = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME, header=HEADER_ROW)
df_raw = df_raw[df_raw['Player'].notna()].copy()
df_raw = df_raw[df_raw['Player'].astype(str).str.strip() != ''].copy()

week_cols = [c for c in df_raw.columns if str(c).startswith('Week')]
print(f'Players found: {len(df_raw)}')
print(f'Week columns : {week_cols}')
print()
df_raw[['Player'] + week_cols]

In [ ]:
# ── CELL 6: Define ETL functions ──────────────────────────────────
import re
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

def tbl(name):
    return f'{PROJECT_ID}.{DATASET_ID}.{name}'

def slugify(name):
    return re.sub(r'[^a-z0-9]+', '_', name.strip().lower()).strip('_')

def parse_display_name(raw):
    """Split 'Name - Nickname' or return (name, None)."""
    if ' - ' in raw:
        parts = raw.split(' - ', 1)
        return parts[0].strip(), parts[1].strip()
    return raw.strip(), None

def bq_load(df, table_name):
    """Batch load DataFrame → BigQuery. Works on free Sandbox tier."""
    job_config = bigquery.LoadJobConfig(write_disposition='WRITE_APPEND')
    job = client.load_table_from_dataframe(df, tbl(table_name), job_config=job_config)
    job.result()
    print(f'  ✅ {table_name}: {len(df)} rows loaded')

def build_players_df(df_raw, season_year):
    player_map = {}
    rows = []
    for raw_name in df_raw['Player'].dropna().unique():
        raw_name = str(raw_name).strip()
        if not raw_name:
            continue
        display_name, nickname = parse_display_name(raw_name)
        player_id = f'{slugify(display_name)}_{season_year}'
        player_map[raw_name] = player_id
        rows.append({
            'player_id':    player_id,
            'display_name': display_name,
            'nickname':     nickname,
            'season_year':  season_year
        })
    return pd.DataFrame(rows), player_map

def build_picks_df(df_raw, season_year, player_map):
    week_cols = [c for c in df_raw.columns if str(c).startswith('Week')]
    rows = []
    for _, row in df_raw.iterrows():
        raw_name = str(row.get('Player', '')).strip()
        if not raw_name or raw_name not in player_map:
            continue
        player_id = player_map[raw_name]
        for col in week_cols:
            week_num = int(re.search(r'\d+', str(col)).group())
            team_picked = row.get(col)
            if pd.isna(team_picked) or str(team_picked).strip() == '':
                team_picked = None
            else:
                team_picked = str(team_picked).strip()
            rows.append({
                'pick_id':     f'{slugify(raw_name)}_{season_year}_w{week_num:02d}',
                'player_id':   player_id,
                'season_year': season_year,
                'week_number': week_num,
                'team_picked': team_picked
            })
    return pd.DataFrame(rows)

print('Functions defined!')

In [ ]:
# ── CELL 7: Load players table ────────────────────────────────────
df_players, player_map = build_players_df(df_raw, SEASON_YEAR)
print('Players to load:')
print(df_players[['player_id', 'display_name', 'nickname']].to_string(index=False))
print()
bq_load(df_players, 'players')

In [ ]:
# ── CELL 8: Load picks table ──────────────────────────────────────
df_picks = build_picks_df(df_raw, SEASON_YEAR, player_map)
print(f'Picks to load: {len(df_picks)} rows ({len(df_raw)} players x 18 weeks)')
print(df_picks.head(10).to_string(index=False))
print()
bq_load(df_picks, 'picks')

In [ ]:
# ── CELL 9: Load weekly results ───────────────────────────────────
# One entry per week listing every team that won that week.
# Re-run this cell anytime to append additional weeks.

winning_teams_2025 = {
    1:  ['Eagles', 'Ravens', 'Bills', 'Chiefs'],  # ← replace with real 2025 winners
    2:  ['Ravens', 'Cardinals', 'Cowboys'],
    # ... fill in all 18 weeks
}

rows = []
for week_num, teams in winning_teams_2025.items():
    for team in teams:
        rows.append({'season_year': SEASON_YEAR, 'week_number': week_num, 'team_won': team.strip()})

bq_load(pd.DataFrame(rows), 'weekly_results')

In [ ]:
# ── CELL 10: Sanity check — leaderboard ──────────────────────────
query = f"""
SELECT
    p.display_name,
    COUNT(*)                                                           AS total_picks,
    SUM(CASE WHEN pk.team_picked = wr.team_won THEN 1 ELSE 0 END)     AS wins,
    ROUND(
        SUM(CASE WHEN pk.team_picked = wr.team_won THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0), 3
    )                                                                  AS win_pct
FROM `{PROJECT_ID}.{DATASET_ID}.picks` pk
JOIN `{PROJECT_ID}.{DATASET_ID}.players` p
    ON pk.player_id = p.player_id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.weekly_results` wr
    ON  pk.season_year = wr.season_year
    AND pk.week_number = wr.week_number
    AND pk.team_picked = wr.team_won
WHERE pk.season_year = {SEASON_YEAR}
GROUP BY p.display_name
ORDER BY wins DESC, win_pct DESC
"""
df_results = client.query(query).to_dataframe()
print(f'{SEASON_YEAR} Leaderboard:')
df_results